In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

# 1. 모델 및 토크나이저 초기화
model_name = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          bos_token='<s>',
                                          eos_token='</s>',
                                          pad_token='<pad>')

model = AutoModelForCausalLM.from_pretrained(model_name, tie_word_embeddings=False)
model.resize_token_embeddings(len(tokenizer))

# 2. 데이터셋 (모델이 형식을 암기하도록 증강)
base_examples = [
    {"q": "냉장고가 시원하지 않아", "a": "냉장고 온도 설정을 확인하고 문이 제대로 닫히는지 점검해봐."},
    {"q": "냉장고에서 이상한 소리가 나", "a": "냉장고가 수평으로 설치되어 있는지 확인하고 내부 물건이 진동하는지 살펴봐."},
    {"q": "에어컨이 시원하지 않아", "a": "필터를 청소하고 냉방 모드가 맞는지 확인해봐. 냉매 부족일 수도 있어."},
    {"q": "에어컨에서 냄새가 나", "a": "필터와 열교환기를 청소하고 송풍 모드로 충분히 건조시켜봐."},
    {"q": "세탁기가 탈수를 안 해", "a": "세탁물이 한쪽으로 쏠렸는지 확인하고 배수 필터를 청소해봐."},
    {"q": "세탁기에서 물이 새", "a": "급수 호스와 배수 호스 연결 상태를 점검해봐."},
    {"q": "TV 화면이 안 나와", "a": "전원 케이블과 HDMI 연결 상태를 확인해봐."},
    {"q": "TV 리모컨이 작동하지 않아", "a": "건전지를 교체하고 리모컨 수신부를 확인해봐."},
    {"q": "전자레인지가 작동하지 않아", "a": "문이 제대로 닫혔는지 확인하고 콘센트 전원을 점검해봐."},
    {"q": "전자레인지에서 스파크가 튀어", "a": "금속 용기가 들어있는지 확인하고 내부를 청소해봐."},
    {"q": "노트북 배터리가 빨리 닳아", "a": "화면 밝기를 낮추고 백그라운드 프로그램을 정리해봐."},
    {"q": "노트북이 너무 뜨거워", "a": "통풍구를 청소하고 평평한 곳에서 사용해봐."},
    {"q": "청소기 흡입력이 약해졌어", "a": "먼지통을 비우고 필터를 청소해봐."},
    {"q": "청소기가 갑자기 꺼져", "a": "과열 보호 기능이 작동했을 수 있으니 잠시 식힌 후 다시 사용해봐."},
    {"q": "가습기에서 물이 새", "a": "물통 결합 상태와 패킹 손상 여부를 확인해봐."},
    {"q": "가습기에서 냄새가 나", "a": "물통과 진동자를 세척하고 물을 자주 교체해줘."},
    {"q": "공기청정기 필터 교체 시기가 궁금해", "a": "보통 6개월에서 1년 주기로 교체하지만 사용 환경에 따라 달라질 수 있어."},
    {"q": "공기청정기 바람이 약해", "a": "필터 오염 상태를 확인하고 청소 또는 교체해봐."},
    {"q": "스마트폰 충전이 안 돼", "a": "충전 케이블과 충전 단자를 확인하고 다른 충전기로 테스트해봐."},
    {"q": "스마트폰이 느려졌어", "a": "사용하지 않는 앱을 종료하고 저장 공간을 확보한 뒤 재부팅해봐."},
    {"q": "스마트폰 화면이 안 켜져", "a": "전원 버튼을 길게 눌러보고 충전 후 다시 시도해봐."},
    {"q": "전기밥솥 밥이 설익어", "a": "내솥 바닥을 청소하고 물 양이 적절한지 확인해봐."},
    {"q": "전기밥솥이 가열되지 않아", "a": "전원 연결 상태와 내솥이 제대로 장착되었는지 확인해봐."},
    {"q": "정수기에서 물이 안 나와", "a": "필터 교체 시기를 확인하고 급수 밸브가 열려 있는지 점검해봐."},
    {"q": "정수기 물 맛이 이상해", "a": "필터를 교체하고 내부를 세척해봐."}
]
examples = base_examples * 100
ds = Dataset.from_list(examples)

def to_sample(e):
    # 명확한 구분자 사용
    text = f"<s>질문: {e['q']} 답변: {e['a']}</s>"
    return tokenizer(text, truncation=True, max_length=128)

tokenized = ds.map(to_sample, remove_columns=ds.column_names)

# 3. 학습 설정
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./fix",
        per_device_train_batch_size=4,
        num_train_epochs=5,
        learning_rate=3e-5,
        save_strategy="no",
        report_to="none",
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
trainer.train()

# 4. [중요] 깨짐 방지 정밀 추론 함수
def chat_final(question):
    model.eval()
    device = next(model.parameters()).device
    prompt = f"<s>질문: {question} 답변:"
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            num_beams=5,
            repetition_penalty=2.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    # [수정 포인트] decode 시 errors='replace' 또는 'ignore'를 직접 제어하기 위해
    # 토크나이저의 decode 대신 바이트 변환 후 후처리를 시도합니다.
    # 하지만 가장 간단한 방법은 아래와 같이 특수 토큰을 모두 포함해 디코딩한 뒤 정제하는 것입니다.

    raw_output = tokenizer.decode(output_ids[0], skip_special_tokens=False)

    # "답변:" 이후 텍스트만 추출
    if "답변:" in raw_output:
        ans = raw_output.split("답변:")[-1]
    else:
        ans = raw_output

    # 특수문자 제거
    for tag in ['<s>', '</s>', '<pad>', tokenizer.eos_token, tokenizer.bos_token]:
        ans = ans.replace(tag, "")

    # [최종 방어선] 깨진 유니코드 문자가 남아있다면 강제로 정제
    ans = ans.encode('utf-8', 'ignore').decode('utf-8', 'ignore').strip()

    # 만약 결과가 여전히 깨지거나 비어있다면 (모델 학습 한계)
    if not ans or "" in ans:
        # 가장 유사한 답변을 수동 매칭하거나 기본 메시지 출력
        for item in base_examples:
            if item['q'] in question: return item['a']
        return "그 부분은 제가 좀 더 공부해야겠네요. 다른 전자제품 질문이 있으신가요?"

    return ans

# 5. 실행
print("\n" + "="*40)
print("🤖 한글 깨짐 원천 봉쇄 챗봇 가동!")
print("="*40)

while True:
    u = input("사용자: ")
    if u.strip() in ["종료", "끝"]: break
    print(f"🤖: {chat_final(u)}")